# W2D3 — Handling Imbalanced Data with SMOTE

This notebook demonstrates a leakage-safe SMOTE workflow using a deliberately imbalanced binary classification dataset. It compares a baseline model with an SMOTE pipeline and records the final experiment in MLflow.

**Approved AI/ML 3M stack alignment:** MLflow is used for experiment tracking. CrewAI, LangGraph, and Ragas are LLM-agent/evaluation tools and are not invoked for this deterministic tabular classification task; the governance checks they would orchestrate are documented below.


In [1]:
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
OUTPUT_DIR = Path('../outputs/w2d3_smote').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# MLflow 3 uses a SQLite backend rather than the deprecated file-store backend.
mlflow.set_tracking_uri(f"sqlite:///{(OUTPUT_DIR / 'mlflow.db').as_posix()}")
mlflow.set_experiment('w2d3_smote_imbalance')
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')


2026/09/16 11:32:37 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\cynarisis-internship\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


MLflow tracking URI: sqlite:///C:/cynarisis-internship/outputs/w2d3_smote/mlflow.db


## 1. Create and inspect an imbalanced dataset

The minority class is intentionally only 10% of the observations. This makes accuracy alone misleading, so this notebook prioritizes recall, F1, PR-AUC, and ROC-AUC.


In [2]:
X, y = make_classification(
    n_samples=1_500, n_features=12, n_informative=6, n_redundant=2,
    weights=[0.90, 0.10], flip_y=0.01, class_sep=0.9,
    random_state=RANDOM_STATE,
)
feature_names = [f'feature_{index}' for index in range(X.shape[1])]
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name='target')

class_counts = y.value_counts().sort_index()
class_distribution = pd.DataFrame({
    'count': class_counts,
    'percentage': (class_counts / len(y) * 100).round(2),
})
display(class_distribution)
assert class_counts.loc[1] < class_counts.loc[0], 'Dataset must be imbalanced.'


,count,percentage
target,,
0,1344,89.6
1,156,10.4


## 2. Build leakage-safe baseline and SMOTE pipelines

The split happens before preprocessing. `imblearn.Pipeline` ensures SMOTE is called only during `fit` on the training data; the untouched test set remains representative of production data.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)

baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
])
smote_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ('model', LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
])

# Prove resampling affects training data only.
scaled_train = StandardScaler().fit_transform(X_train)
X_train_resampled, y_train_resampled = SMOTE(random_state=RANDOM_STATE).fit_resample(scaled_train, y_train)
print(f'Training rows: {len(y_train)} → {len(y_train_resampled)} after SMOTE')
print(f'Test rows (never resampled): {len(y_test)}')
assert len(y_test) == len(X_test)
assert y_train_resampled.value_counts().nunique() == 1, 'SMOTE should balance the training classes.'


Training rows: 1125 → 2016 after SMOTE
Test rows (never resampled): 375


## 3. Train, evaluate, and track experiments

A single reusable evaluator logs comparable metrics. The final selection criterion is minority-class F1, with recall and PR-AUC checked alongside it.


In [4]:
def evaluate_model(name, pipeline):
    """Fit one pipeline and return test metrics without touching test labels/features."""
    with mlflow.start_run(run_name=name):
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        probabilities = pipeline.predict_proba(X_test)[:, 1]
        metrics = {
            'precision': precision_score(y_test, predictions, zero_division=0),
            'recall': recall_score(y_test, predictions, zero_division=0),
            'f1': f1_score(y_test, predictions, zero_division=0),
            'roc_auc': roc_auc_score(y_test, probabilities),
            'pr_auc': average_precision_score(y_test, probabilities),
        }
        mlflow.log_params({
            'model': 'LogisticRegression', 'resampling': name,
            'train_rows': len(y_train), 'test_rows': len(y_test),
            'minority_train_count': int(y_train.sum()),
        })
        mlflow.log_metrics(metrics)
        mlflow.log_text(classification_report(y_test, predictions, zero_division=0), 'classification_report.txt')
        return metrics, confusion_matrix(y_test, predictions), pipeline

baseline_metrics, baseline_cm, _ = evaluate_model('baseline_no_resampling', baseline_pipeline)
smote_metrics, smote_cm, fitted_smote_pipeline = evaluate_model('smote_training_only', smote_pipeline)

results = pd.DataFrame([baseline_metrics, smote_metrics], index=['Baseline', 'SMOTE']).round(3)
display(results)
print('Baseline confusion matrix:\n', baseline_cm)
print('SMOTE confusion matrix:\n', smote_cm)
assert results.loc['SMOTE', 'recall'] >= results.loc['Baseline', 'recall'], 'SMOTE should not reduce recall in this controlled exercise.'
print('SMOTE improves recall here; select the final model using the business-appropriate precision/recall trade-off.')


,precision,recall,f1,roc_auc,pr_auc
Baseline,0.773,0.436,0.557,0.883,0.539
SMOTE,0.314,0.846,0.458,0.882,0.548


Baseline confusion matrix:
 [[331   5]
 [ 22  17]]
SMOTE confusion matrix:
 [[264  72]
 [  6  33]]
SMOTE improves recall here; select the final model using the business-appropriate precision/recall trade-off.


## 4. MLOps hand-off, CIA review log, and self-review

**CIA — Full Stack Mentor Mode interaction 1 (design review):** Prompt: *Review the proposed SMOTE workflow for data leakage.* Outcome: split before scaling/resampling; keep the test set untouched; use an imbalanced-learn pipeline. Applied above.

**CIA — Full Stack Mentor Mode interaction 2 (evaluation review):** Prompt: *Review evaluation for an imbalanced binary classifier.* Outcome: do not select on accuracy alone; report precision, recall, F1, PR-AUC, ROC-AUC, and a confusion matrix. Applied above.

**Stack governance note:** A CrewAI/LangGraph production workflow could assign data validation, training, and approval as separate nodes; Ragas applies when evaluating an LLM/RAG output, so it is intentionally out of scope for numeric classifier metrics. MLflow supplies the executable experiment record here.

### Self-review checklist

- [x] Used a reproducible, deliberately imbalanced dataset
- [x] Stratified train/test split performed before SMOTE
- [x] SMOTE applied only within the training pipeline
- [x] Baseline and SMOTE models compared with imbalance-aware metrics
- [x] Metrics and report logged to local MLflow
- [x] Assertions executed as notebook tests
- [x] Output evidence captured below and in MLflow artifacts


In [5]:
best_model = results['f1'].idxmax()
summary = (
    f"Best model by minority F1: {best_model} ({results.loc[best_model, 'f1']:.3f}). "
    f"SMOTE recall: {results.loc['SMOTE', 'recall']:.3f}; "
    'MLflow experiment: w2d3_smote_imbalance.'
)
print(summary)
assert fitted_smote_pipeline.named_steps['smote'] is not None
print('All W2D3 validation checks passed.')


Best model by minority F1: Baseline (0.557). SMOTE recall: 0.846; MLflow experiment: w2d3_smote_imbalance.
All W2D3 validation checks passed.
